# K-means Clustering + PCA — CESA3 / HvGlu3 Complex
**University of Jordan — CESA Project**

Same methodology as `CESA1_kmeans_v2.ipynb`, applied to CESA3, so the two
systems' conformational landscapes can be directly compared.

This notebook performs:
1. Load merged trajectory (`CESA3_all_backbone.xtc`)
2. Featurize using Ca pairwise distances (every 3rd residue to avoid memory crash)
3. PCA to reduce dimensions
4. Elbow method to find optimal number of clusters
5. K-means clustering
6. Save representative PDB for each cluster
7. Save cluster populations

**Note:** Run with `Python (md_analysis)` kernel

In [ ]:
# ============================================================
# CELL 1 — Imports
# ============================================================
%matplotlib inline

import mdtraj as md
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from itertools import combinations

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist

plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family'] = 'sans-serif'
sns.set_style('whitegrid')
print('All imports successful!')

In [ ]:
# ============================================================
# CELL 2 — File paths (CESA3 — process_CESA3 folder)
# ============================================================
from pathlib import Path

RUN_DIR    = Path('/Volumes/SAMSUNG/CESAproject/MD_after_docking/CESA3/process_CESA3')
TOPOLOGY   = RUN_DIR / 'R1' / 'backbone.pdb'
TRAJECTORY = RUN_DIR / 'CESA3_all_backbone.xtc'
OUT_DIR    = RUN_DIR / 'KMeans'

OUT_DIR.mkdir(exist_ok=True)

# Verify files exist
for f in [TOPOLOGY, TRAJECTORY]:
    status = '\u2713' if f.exists() else '\u2717 MISSING'
    size   = f'{f.stat().st_size / 1e6:.1f} MB' if f.exists() else ''
    print(f'  {status}  {f.name}  {size}')

In [ ]:
# ============================================================
# CELL 3 — Load trajectory
# ============================================================

print(f'Loading trajectory...')
traj = md.load(str(TRAJECTORY), top=str(TOPOLOGY))

print(traj)
print(f'\nFrames    : {traj.n_frames}')
print(f'Atoms     : {traj.n_atoms}')
print(f'Residues  : {traj.n_residues}')

# Confirm chain structure
chains = list(traj.topology.chains)
print(f'\nChain structure:')
for chain in chains:
    residues = list(chain.residues)
    print(f'  Chain {chain.index}: {len(residues)} residues '
          f'({residues[0].name}{residues[0].resSeq} -> {residues[-1].name}{residues[-1].resSeq})')

In [ ]:
# ============================================================
# CELL 4 — Featurization using Ca pairwise distances
#
# Same memory-saving approach as CESA1: every 3rd Ca atom.
# ============================================================

# Select Ca atoms
ca_indices = traj.topology.select('name CA')
print(f'Total Ca atoms: {len(ca_indices)}')

# Use every 3rd Ca
ca_reduced = ca_indices[::3]
print(f'Reduced Ca atoms (every 3rd): {len(ca_reduced)}')

# Compute pairs
ca_pairs = np.array(list(combinations(ca_reduced, 2)))
print(f'Number of pairs: {len(ca_pairs)}')

# Estimate memory
mem_gb = len(ca_pairs) * traj.n_frames * 8 / 1e9
print(f'Estimated memory: {mem_gb:.1f} GB')

print('\nComputing pairwise distances...')
X = md.compute_distances(traj, ca_pairs)
print(f'Feature matrix shape: {X.shape}  <- (frames x pairs)')
print('Featurization complete!')

In [ ]:
# ============================================================
# CELL 5 — PCA
# ============================================================

print('Running PCA...')
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X)

data1 = pd.DataFrame(X_pca, columns=['PC1', 'PC2', 'PC3'])

explained = pca.explained_variance_ratio_ * 100
print(f'PC1: {explained[0]:.1f}%')
print(f'PC2: {explained[1]:.1f}%')
print(f'PC3: {explained[2]:.1f}%')
print(f'Total (3 PCs): {sum(explained):.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(data1['PC1'], data1['PC2'], alpha=0.4, s=5, c='steelblue')
axes[0].set_xlabel(f'PC1 ({explained[0]:.1f}%)', fontsize=13)
axes[0].set_ylabel(f'PC2 ({explained[1]:.1f}%)', fontsize=13)
axes[0].set_title('PCA \u2014 PC1 vs PC2', fontsize=14, weight='bold')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

axes[1].scatter(data1['PC1'], data1['PC3'], alpha=0.4, s=5, c='darkorange')
axes[1].set_xlabel(f'PC1 ({explained[0]:.1f}%)', fontsize=13)
axes[1].set_ylabel(f'PC3 ({explained[2]:.1f}%)', fontsize=13)
axes[1].set_title('PCA \u2014 PC1 vs PC3', fontsize=14, weight='bold')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle('PCA of C\u03b1 Pairwise Distances \u2014 CESA3',
             fontsize=15, weight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'PCA_plot.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: PCA_plot.png')

In [ ]:
# ============================================================
# CELL 6 — Elbow method
# ============================================================

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(data1[['PC1', 'PC2', 'PC3']])

inertia = []
k_range = range(1, 11)

print('Running elbow method (k=1 to 10)...')
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    km.fit(X_scaled)
    inertia.append(km.inertia_)
    print(f'  k={k:2d}: inertia={km.inertia_:.2f}')

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(k_range), inertia, marker='o', linewidth=2,
        color='steelblue', markersize=8)
ax.set_xlabel('Number of Clusters (K)', fontsize=13)
ax.set_ylabel('Inertia', fontsize=13)
ax.set_title('Elbow Method \u2014 CESA3',
             fontsize=14, weight='bold')
ax.set_xticks(list(k_range))
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(OUT_DIR / 'Elbow_plot.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: Elbow_plot.png')
print('\n>>> Look at the plot and choose K at the elbow bend')
print('>>> IMPORTANT: CESA3 has only 4 source trajectories (vs CESA1\'s 8).')
print('    Do not assume K=4 carries over from CESA1 \u2014 choose K based on')
print('    THIS elbow plot. With fewer underlying trajectories, CESA3 may')
print('    genuinely support fewer (or differently-shaped) clusters.')

In [ ]:
# ============================================================
# CELL 7 — K-means clustering
# *** CHANGE n_clusters BASED ON THIS SYSTEM'S ELBOW PLOT ABOVE ***
# Do not copy CESA1's K=4 without checking — verify against the
# CESA3 elbow plot first.
# ============================================================

n_clusters = 4   # <-- change this after looking at the CESA3 elbow plot

print(f'Running K-means with K={n_clusters}...')

kmeans         = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
clusters       = kmeans.fit_predict(X_scaled)
data1['Cluster'] = clusters
centers_scaled = kmeans.cluster_centers_
centers        = scaler.inverse_transform(centers_scaled)

print('\nCluster centers (PC1, PC2, PC3):')
for i, c in enumerate(centers):
    count = (clusters == i).sum()
    pct   = count / len(clusters) * 100
    print(f'  Cluster {i+1}: PC1={c[0]:.3f}  PC2={c[1]:.3f}  PC3={c[2]:.3f}  '
          f'({count} frames, {pct:.1f}%)')

# Plot
cmap   = plt.cm.tab10  # ✓ New style
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (xkey, ykey) in zip(axes, [('PC1','PC2'), ('PC1','PC3')]):
    scatter = ax.scatter(
        data1[xkey], data1[ykey],
        c=data1['Cluster'], cmap='tab10',
        vmin=0, vmax=n_clusters-1,
        s=10, alpha=0.6
    )
    xi = ['PC1','PC2','PC3'].index(xkey)
    yi = ['PC1','PC2','PC3'].index(ykey)
    ax.scatter(
        centers[:, xi], centers[:, yi],
        c='red', s=250, marker='X',
        edgecolors='black', linewidths=1.5,
        label='Centroids', zorder=5
    )
    for i, c in enumerate(centers):
        ax.text(c[xi], c[yi]+0.2, f'C{i+1}',
                fontsize=10, weight='bold', ha='center')
    ax.set_xlabel(f'{xkey} ({explained[xi]:.1f}%)', fontsize=12)
    ax.set_ylabel(f'{ykey} ({explained[yi]:.1f}%)', fontsize=12)
    ax.set_title(f'{xkey} vs {ykey}', fontsize=13, weight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.colorbar(scatter, ax=axes[1], label='Cluster')
plt.suptitle(f'K-means Clustering (K={n_clusters}) \u2014 CESA3',
             fontsize=14, weight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'KMeans_clusters.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: KMeans_clusters.png')

In [ ]:
# ============================================================
# CELL 8 — Representative frame per cluster + save PDB
# ============================================================

distances      = cdist(X_scaled, centers_scaled)
closest_frames = np.argmin(distances, axis=0)

print('Representative frames (closest to centroid):')
for i, frame in enumerate(closest_frames):
    count = (clusters == i).sum()
    pct   = count / len(clusters) * 100
    print(f'  Cluster {i+1}: frame {frame}  ({count} frames, {pct:.1f}% of trajectory)')

# Save PDB files
pdb_dir = OUT_DIR / 'pdbs'
pdb_dir.mkdir(exist_ok=True)

print('\nSaving representative PDB files...')
for i, frame in enumerate(closest_frames):
    output_name = str(pdb_dir / f'cluster_{i+1}_frame_{frame}.pdb')
    traj[frame].save_pdb(output_name)
    print(f'  Saved: cluster_{i+1}_frame_{frame}.pdb')

print(f'\nAll PDB files saved in {pdb_dir}')

In [ ]:
# ============================================================
# CELL 9 — Cluster populations
# ============================================================

population = data1['Cluster'].value_counts().sort_index()

population_df = pd.DataFrame({
    'Cluster'   : population.index + 1,
    'Population': population.values,
    'Percentage': (population.values / len(data1) * 100).round(2)
})

print('Cluster populations:')
print(population_df.to_string(index=False))

# Bar plot
fig, ax = plt.subplots(figsize=(8, 5))
colors_bar = [plt.cm.tab10(i / n_clusters) for i in range(n_clusters)]
bars = ax.bar(
    population_df['Cluster'],
    population_df['Percentage'],
    color=colors_bar, edgecolor='black', linewidth=0.8
)

# Add percentage labels on bars
for bar, pct in zip(bars, population_df['Percentage']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.3,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=10)

ax.set_xlabel('Cluster', fontsize=13)
ax.set_ylabel('Population (%)', fontsize=13)
ax.set_title(f'Cluster Populations (K={n_clusters}) \u2014 CESA3',
             fontsize=14, weight='bold')
ax.set_xticks(population_df['Cluster'])
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(OUT_DIR / 'cluster_populations.png', dpi=300, bbox_inches='tight')
plt.show()

# Save CSVs
population_df.to_csv(OUT_DIR / 'cluster_populations.csv', index=False)
data1.to_csv(OUT_DIR / 'PCA_clusters.csv', index=False)

print('\nAll outputs saved to:', OUT_DIR)
print('  PCA_plot.png')
print('  Elbow_plot.png')
print('  KMeans_clusters.png')
print('  cluster_populations.png')
print('  cluster_populations.csv')
print('  PCA_clusters.csv')
print('  pdbs/cluster_N_frame_M.pdb')

In [ ]:
# ============================================================
# CELL 10 — Which source trajectory (T1-T4) does each cluster
# come from? CESA3-specific addition: since the merged file is
# T1->T2->T3->T4 laid end to end (see process_trajectories_CESA3.sh,
# steps 11-12), frame index directly tells you which T a frame
# belongs to. This lets you check whether each cluster is dominated
# by a single trajectory (consistent with "T1 found one pose, T2
# found another") or mixed across several Ts.
# ============================================================

# EDIT these boundaries if your actual per-T frame counts differ —
# check via: print(traj.n_frames) and compare against 4x expected length
n_frames_total = traj.n_frames
frames_per_T = n_frames_total // 4   # assumes T1-T4 are equal length

def frame_to_trajectory(frame_idx):
    t_num = min(frame_idx // frames_per_T, 3) + 1
    return f'T{t_num}'

data1['source_trajectory'] = data1.index.map(frame_to_trajectory)

print(f'Assuming {frames_per_T} frames per trajectory '
      f'({n_frames_total} total / 4)\n')

crosstab = pd.crosstab(data1['Cluster'] + 1, data1['source_trajectory'])
crosstab_pct = crosstab.div(crosstab.sum(axis=1), axis=0) * 100

print('Cluster composition by source trajectory (% of cluster):')
print(crosstab_pct.round(1).to_string())

print('\nInterpretation:')
print('  - A cluster dominated by ONE trajectory (e.g. >80% from T2)')
print('    suggests that pose is specific to that single run.')
print('  - A cluster with frames spread across MULTIPLE trajectories')
print('    suggests that pose is reproducible / shared across runs.')

In [ ]:
# ============================================================
# CELL 11 — Combine 3 figures into one publication-quality figure
# ============================================================
import matplotlib.image as mpimg

elbow_img   = mpimg.imread(OUT_DIR / 'Elbow_plot.png')
kmeans_img  = mpimg.imread(OUT_DIR / 'KMeans_clusters.png')
pop_img     = mpimg.imread(OUT_DIR / 'cluster_populations.png')

fig = plt.figure(figsize=(16, 14))

ax_a = fig.add_subplot(2, 2, 1)
ax_a.imshow(elbow_img)
ax_a.axis('off')
ax_a.text(-0.05, 1.05, 'A', transform=ax_a.transAxes,
          fontsize=16, weight='bold', va='top', ha='right')

ax_b = fig.add_subplot(2, 2, 2)
ax_b.imshow(pop_img)
ax_b.axis('off')
ax_b.text(-0.05, 1.05, 'B', transform=ax_b.transAxes,
          fontsize=16, weight='bold', va='top', ha='right')

ax_c = fig.add_subplot(2, 1, 2)
ax_c.imshow(kmeans_img)
ax_c.axis('off')
ax_c.text(-0.02, 1.02, 'C', transform=ax_c.transAxes,
          fontsize=16, weight='bold', va='top', ha='right')

fig.suptitle(
    'K-means Clustering Analysis \u2014 CESA3',
    fontsize=15, weight='bold', y=1.01
)

plt.tight_layout()

out_path = OUT_DIR / 'KMeans_combined_figure.png'
fig.savefig(out_path, dpi=300, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RUN_DIR = Path('/Volumes/SAMSUNG/CESAproject/MD_after_docking/CESA3/process_CESA3')
OUT_DIR = RUN_DIR / 'KMeans'

data1 = pd.read_csv(OUT_DIR / 'PCA_clusters.csv')
print(data1['Cluster'].value_counts().sort_index())

explained = [32.9, 19.3, 15.5]
closest_frames = {0: 723, 1: 1328, 2: 199, 3: 1058}  # from Cell 8, 0-indexed to match Cluster column

fig, ax = plt.subplots(figsize=(9, 7))

cluster_colors = {0: '#1f4e8c', 1: '#2e8b57', 2: '#8b5a2b', 3: '#808080'}
point_colors = data1['Cluster'].map(cluster_colors)

ax.scatter(data1['PC1'], data1['PC2'], c=point_colors, s=50, alpha=0.6)

for i, frame in closest_frames.items():
    x = data1['PC1'].iloc[frame]
    y = data1['PC2'].iloc[frame]
    ax.scatter(x, y, c=cluster_colors[i], s=220, alpha=0.95,
               edgecolors='black', linewidths=1.5, zorder=5)
    ax.annotate(f'Cluster {i+1}', (x, y),
                textcoords="offset points", xytext=(10, 10),
                fontsize=10, weight='bold')

ax.set_xlabel(f'PC1 ({explained[0]:.1f}%)', fontsize=13)
ax.set_ylabel(f'PC2 ({explained[1]:.1f}%)', fontsize=13)
ax.set_title('K-means Clustering — CESA3', weight='bold', fontsize=16)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

legend_elements = [plt.Line2D([0], [0], marker='o', color='w',
                    markerfacecolor=cluster_colors[c], markersize=10,
                    label=f'Cluster {c+1}') for c in [0, 1, 2, 3]]
ax.legend(handles=legend_elements, loc='best', frameon=True)

fig.tight_layout()
fig.savefig(OUT_DIR / 'CESA3_KMeans_custom_colors.png', dpi=300, bbox_inches='tight')
plt.show()